In [4]:
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score
import mlflow
from pathlib import Path

# ==========================================
# 1. SETUP PATH & LOAD DATA
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("07b_Fair_Apple_to_Apple_Comparison")

print("🔍 1. Memuat Data Latih dan Uji...")
train_df = pd.read_csv(root_path / "Data/processed/train_selected_features.csv")
test_df = pd.read_csv(root_path / "Data/split/test_data.csv")

features = ['TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 
            'VCL1', 'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL11', 'VCL12', 
            'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education', 'urban', 'gender', 'engnat', 'age', 
            'religion', 'orientation', 'race', 'voted', 'married']
targets = ['risk_depression', 'risk_anxiety', 'risk_stress']

X_train, Y_train = train_df[features], train_df[targets]
X_test, Y_test = test_df[features], test_df[targets]

# ==========================================
# 2. HYPERPARAMETER TUNING YANG ADIL
# ==========================================
print("⚖️ 2. Memulai Tuning Adil untuk Kedua Model...")

# -- A. Tuning Random Forest --
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
base_rf = RandomForestClassifier(random_state=42)
print("   -> Mencari parameter terbaik untuk Random Forest...")
rf_search = RandomizedSearchCV(base_rf, rf_param_grid, n_iter=10, cv=3, scoring='f1_macro', n_jobs=-1, random_state=42)
# Kita tuning base model-nya dulu menggunakan salah satu target (agar cepat), misal target pertama
rf_search.fit(X_train, Y_train['risk_depression']) 
tuned_rf = rf_search.best_estimator_

# -- B. Tuning XGBoost --
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0]
}
base_xgb = xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)
print("   -> Mencari parameter terbaik untuk XGBoost...")
xgb_search = RandomizedSearchCV(base_xgb, xgb_param_grid, n_iter=10, cv=3, scoring='f1_macro', n_jobs=-1, random_state=42)
xgb_search.fit(X_train, Y_train['risk_depression'])
tuned_xgb = xgb_search.best_estimator_

# ==========================================
# 3. PERTANDINGAN CLASSIFIER CHAIN
# ==========================================
print("\n🥊 3. Memulai Pertandingan Multi-label (Classifier Chain)...")

model_rf_chain = ClassifierChain(tuned_rf, order='random', random_state=42)
model_xgb_chain = ClassifierChain(tuned_xgb, order='random', random_state=42)

def evaluate_fair_model(model, name):
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)
    return f1_score(Y_test, Y_pred, average='macro')

with mlflow.start_run(run_name="Fair_Tuned_Comparison"):
    rf_macro = evaluate_fair_model(model_rf_chain, "Tuned Random Forest Chain")
    xgb_macro = evaluate_fair_model(model_xgb_chain, "Tuned XGBoost Chain")
    
    print("\n" + "="*50)
    print("🏆 HASIL PERTANDINGAN APPLE-TO-APPLE 🏆")
    print(f"1. Tuned XGBoost (Chain)       - Macro F1: {xgb_macro:.4f}")
    print(f"2. Tuned Random Forest (Chain) - Macro F1: {rf_macro:.4f}")
    print("="*50)
    
    if xgb_macro > rf_macro:
        print("\n✅ KESIMPULAN: Setelah sama-sama di-tuning, XGBoost kembali merebut takhtanya!")
    else:
        print("\n⚠️ KESIMPULAN: Random Forest memang MONSTER di dataset ini. Dia menang mutlak secara adil.")

🔍 1. Memuat Data Latih dan Uji...
⚖️ 2. Memulai Tuning Adil untuk Kedua Model...
   -> Mencari parameter terbaik untuk Random Forest...
   -> Mencari parameter terbaik untuk XGBoost...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [20:55:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



🥊 3. Memulai Pertandingan Multi-label (Classifier Chain)...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [20:55:37] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [20:55:38] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



🏆 HASIL PERTANDINGAN APPLE-TO-APPLE 🏆
1. Tuned XGBoost (Chain)       - Macro F1: 0.7841
2. Tuned Random Forest (Chain) - Macro F1: 0.7867

⚠️ KESIMPULAN: Random Forest memang MONSTER di dataset ini. Dia menang mutlak secara adil.
